# Пайплайн: INRIA + NY building + zone (UBC val/test) — **шаг 08, финальный**

Три модели на аэрофото UBC:

1. **INRIA** (`inria_building_segmenter.pth`, 04) — бинарная маска зданий, sliding window.
2. **NY building** (`ny_building_ubc.pth` или `ny_building_classifier.pth`, 05 / 06 fine-tune) — тип здания: residential / commercial / industrial.
3. **Zone** (`convnext_best.pth`, 03) — карта застройки: dense/sparse/commercial/industrial.
4. **Merge** — residential уточняется по zone-карте.

**Оценка:** GT из UBC COCO только для метрик (mask IoU, macro F1 building, merge summary).

Сплиты crop'ов: `data/processed_ubc/split.csv` (test = исходный UBC val).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

sys.path.insert(0, str(Path('..').resolve() / 'src'))

from building_masks import draw_building_masks
from dataset import CLASSES as ZONE_CLASSES
from merge_maps import draw_merged_masks, merge_summary
from model_convnext import build_convnext_tiny
from model_segmentation import build_convnext_segmenter
from pipeline_ubc import (
    evaluate_building_classification,
    run_pipeline_on_tile,
    ubc_tile_names_for_eval_split,
    zone_overlay,
)
from utils import (
    CONVNEXT_TINY,
    INRIA_SEGMENTATION,
    MODELS_DIR,
    NY_BUILDING,
    NY_BUILDING_CLASSES,
    PLOT_DPI,
    REPORTS_DIR,
    UBC_RAW_DIR,
    ensure_dirs,
)

ensure_dirs()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Устройство:', device)

## Загрузка моделей

In [ ]:
zone_ckpt = MODELS_DIR / CONVNEXT_TINY.model_filename
building_ckpt = MODELS_DIR / NY_BUILDING.model_filename
inria_ckpt = MODELS_DIR / INRIA_SEGMENTATION.model_filename

for name, path in [
    ('zone', zone_ckpt),
    ('building', building_ckpt),
    ('inria', inria_ckpt),
]:
    if not path.exists():
        raise FileNotFoundError(f'Нет {name}: {path}')

zone_model = build_convnext_tiny(num_classes=len(ZONE_CLASSES), freeze_backbone=False)
zone_model.load_state_dict(torch.load(zone_ckpt, map_location=device, weights_only=True))
zone_model.to(device).eval()
print('Zone:', zone_ckpt.name)

building_model = build_convnext_tiny(num_classes=len(NY_BUILDING_CLASSES), freeze_backbone=False)
building_model.load_state_dict(torch.load(building_ckpt, map_location=device, weights_only=True))
building_model.to(device).eval()
print('Building (NY):', building_ckpt.name)

segmenter_model = build_convnext_segmenter(pretrained=False, freeze_encoder=False)
segmenter_model.load_state_dict(torch.load(inria_ckpt, map_location=device, weights_only=True))
segmenter_model.to(device).eval()
print('INRIA:', inria_ckpt.name)

## Конфигурация val / test

Тайлы лежат в `data/raw/ubc/val/`. Сплит crop'ов — в `processed_ubc/split.csv`.

In [ ]:
UBC_IMAGE_SPLIT = 'val'  # каталог raw: ubc/val/*.tif
EVAL_SPLIT = 'test'      # 'val' или 'test' по split.csv
DEMO_TILE_COUNT = 3
MIN_COMPONENT_AREA = 32
MASK_THRESHOLD = 0.5
MATCH_IOU = 0.2

eval_tiles = ubc_tile_names_for_eval_split(EVAL_SPLIT)
print(f'Тайлов для {EVAL_SPLIT}:', len(eval_tiles))

if not eval_tiles:
    eval_tiles = sorted(p.name for p in (UBC_RAW_DIR / UBC_IMAGE_SPLIT).glob('*_RGB.tif'))
    print('Fallback: все val-тайлы:', len(eval_tiles))

demo_tiles = eval_tiles[:DEMO_TILE_COUNT]
for name in demo_tiles:
    print(' ', name)

## Запуск пайплайна на демо-тайлах

In [ ]:
FULL_PIPELINE_DIR = REPORTS_DIR / 'full_pipeline'
FULL_PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

results = []
for tile_name in demo_tiles:
    result = run_pipeline_on_tile(
        tile_name,
        split=UBC_IMAGE_SPLIT,
        zone_model=zone_model,
        segmenter_model=segmenter_model,
        building_model=building_model,
        device=str(device),
        min_component_area=MIN_COMPONENT_AREA,
        mask_threshold=MASK_THRESHOLD,
    )
    results.append(result)
    print(f"{tile_name}: pred={len(result.buildings)} gt={len(result.gt_buildings)} mask_iou={result.mask_iou:.3f}")

print('Готово:', len(results))

## Визуализация: zone | INRIA mask | building pred | merge

In [ ]:
for result in results:
    image = result.image
    zone_vis = zone_overlay(image, result.zone_map)
    building_vis = draw_building_masks(image, result.buildings, use_predicted=True)
    merged_vis = draw_merged_masks(image, result.buildings)

    inria_vis = image.copy()
    if result.pred_mask is not None:
        overlay = np.zeros((*result.pred_mask.shape, 4), dtype=np.uint8)
        overlay[result.pred_mask.astype(bool)] = (255, 80, 80, 140)
        inria_vis = Image.alpha_composite(image.convert('RGBA'), Image.fromarray(overlay, 'RGBA')).convert('RGB')

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    panels = [
        (zone_vis, 'zone'),
        (inria_vis, 'INRIA mask'),
        (building_vis, 'NY building'),
        (merged_vis, 'merge'),
    ]
    for ax, (img, title) in zip(axes, panels):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis('off')
    fig.suptitle(result.tile_name, fontsize=11)
    plt.tight_layout()
    out_path = FULL_PIPELINE_DIR / f"{Path(result.tile_name).stem}.png"
    plt.savefig(out_path, dpi=PLOT_DPI, bbox_inches='tight')
    plt.show()
    print('Сохранено:', out_path)

## Метрики на всём eval-сплите

In [ ]:
rows = []
for tile_name in eval_tiles:
    result = run_pipeline_on_tile(
        tile_name,
        split=UBC_IMAGE_SPLIT,
        zone_model=zone_model,
        segmenter_model=segmenter_model,
        building_model=building_model,
        device=str(device),
        min_component_area=MIN_COMPONENT_AREA,
        mask_threshold=MASK_THRESHOLD,
    )
    cls_metrics = evaluate_building_classification(result, iou_threshold=MATCH_IOU)
    rows.append({
        'tile': tile_name,
        'pred_buildings': len(result.buildings),
        'gt_buildings': len(result.gt_buildings),
        'mask_iou': result.mask_iou,
        'matched': cls_metrics['matched'],
        'building_acc': cls_metrics['accuracy'],
        'building_macro_f1': cls_metrics['macro_f1'],
        'merge': merge_summary(result.buildings),
    })

metrics_df = pd.DataFrame(rows)
print(f"=== Сводка ({EVAL_SPLIT}, n={len(metrics_df)}) ===")
print(f"mask IoU (mean): {metrics_df['mask_iou'].mean():.4f}")
print(f"building macro F1 (mean по тайлам с match): "
      f"{metrics_df.loc[metrics_df['matched']>0, 'building_macro_f1'].mean():.4f}")
print(f"building acc (mean): {metrics_df.loc[metrics_df['matched']>0, 'building_acc'].mean():.4f}")
print(metrics_df.head(10).to_string())

metrics_path = REPORTS_DIR / f'pipeline_ubc_{EVAL_SPLIT}_metrics.csv'
metrics_df.to_csv(metrics_path, index=False)
print('CSV:', metrics_path)

## Ограничения

- INRIA обучен на других городах — mask IoU на Munich UBC может быть ниже, чем на INRIA test.
- NY building обучен на Zenodo small (США) — тип здания на UBC без дообучения может ошибаться.
- Zone-карта зональная (sliding window), не кадастровая граница.
- Instances из INRIA — connected components; мелкие объекты и склеенные маски влияют на matching.
- Residential без dense/sparse в zone остаётся `residential` (нерешённый merge).